In [8]:
import numpy as np
import sionna.rt as rt

from sionna.rt import (
    ITURadioMaterial,
    PathSolver,
    PlanarArray,
    Receiver,
    Transmitter,
    load_scene,
)

CARRIER_FREQUENCY = 2.4e9
SOLVER_SEED = 41

In [9]:
scene = load_scene(
    rt.scene.simple_reflector,
    merge_shapes=False,
)

scene.frequency = CARRIER_FREQUENCY
scene.tx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization="V",
)

scene.rx_array = scene.tx_array

tx = Transmitter(
    name="tx",
    position=[-2.0, 0.0, 1.0],
)

rx = Receiver(
    name="rx",
    position=[2.0, 0.0, 1.0],
)

scene.add(tx)
scene.add(rx)
tx.look_at(rx)

In [10]:
path_solver = PathSolver()


def extract_channel_response(paths):
    coefficients, delays = paths.cir(
        normalize_delays=False,
        out_type="numpy",
    )

    # Sionna RT CIR shape:
    # [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_steps]
    coefficients = np.asarray(coefficients)[0, 0, 0, 0, :, 0]

    # Sionna RT delay shape:
    # [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths]
    delays = np.asarray(delays)[0, 0, 0, 0, :]

    order = np.argsort(delays)
    return coefficients[order], delays[order]

In [11]:
paths_los = path_solver(
    scene=scene,
    max_depth=0,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    refraction=False,
    synthetic_array=False,
    seed=SOLVER_SEED,
)

los_response = extract_channel_response(paths_los)
print("Direct-path delay [ns]:", los_response[1] * 1e9)
print("Direct-path amplitude:", np.abs(los_response[0]))
print("Direct-path phase [rad]:", np.angle(los_response[0]))

Direct-path delay [ns]: [13.342564]
Direct-path amplitude: [0.00248508]
Direct-path phase [rad]: [-0.1391905]


In [12]:
soil_material = ITURadioMaterial(
    name="medium-dry-ground",
    itu_type="medium_dry_ground",
    thickness=0.10,
)

scene.add(soil_material)
scene.objects["reflector"].radio_material = soil_material.name

paths_material = path_solver(
    scene=scene,
    max_depth=1,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=True,
    synthetic_array=False,
    seed=SOLVER_SEED,
)

material_response = extract_channel_response(paths_material)
print("Material-case delay [ns]:", material_response[1] * 1e9)
print("Material-case amplitude:", np.abs(material_response[0]))
print("Material-case phase [rad]:", np.angle(material_response[0]))

Material-case delay [ns]: [13.342564 14.917439]
Material-case amplitude: [0.00248508 0.00058781]
Material-case phase [rad]: [-0.1391905   0.97610575]


In [13]:
channelResponse = {
    "direct": los_response,
    "medium_dry_ground": material_response,
}

for case_name, (coefficients, delays) in channelResponse.items():
    print(case_name)
    for coefficient, delay in zip(coefficients, delays):
        print(
            f"delay={delay * 1e9:.3f} ns, "
            f"amplitude={abs(coefficient):.6f}, "
            f"phase={np.angle(coefficient):.6f} rad"
        )

direct
delay=13.343 ns, amplitude=0.002485, phase=-0.139190 rad
medium_dry_ground
delay=13.343 ns, amplitude=0.002485, phase=-0.139190 rad
delay=14.917 ns, amplitude=0.000588, phase=0.976106 rad


rendering

In [14]:
from pathlib import Path
from sionna.rt import Camera

camera = Camera(
    position=[0.0, -6.0, 3.0],
    look_at=[0.0, 0.0, 1.0],
)

output_path = Path(
    r"C:\Users\jaspe\OneDrive\Documents\sionna-rf-simulation"
) / "results" / "figures" / "simple_reflector_paths.png"

scene.render_to_file(
    camera=camera,
    filename=str(output_path),
    paths=paths_material,
    resolution=[800, 600],
    num_samples=128,
)

print(f"Saved to: {output_path}")

Saved to: C:\Users\jaspe\OneDrive\Documents\sionna-rf-simulation\results\figures\simple_reflector_paths.png
